In [8]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型
# ============================================================
!pip install -q groq gradio

import sqlite3
import pandas as pd
from IPython.display import display
from google.colab import userdata
from groq import Groq
import gradio as gr

# 🔴 金鑰只能從左側「🔑 密鑰（Secrets）」讀取，絕對不可以寫成 api_key="gsk_..."
#    寫死的金鑰一旦存到 GitHub，就等於公開給全世界使用。
#    也不要 print 出金鑰，因為輸出結果也會一起存到 GitHub。
try:
    client = Groq(api_key=userdata.get('SDDB_Lecture05'))
except Exception as e:
    print("❌ 讀不到金鑰。請檢查：")
    print("   1. 左側 🔑 是否新增了名稱為 GROQ_API_KEY 的密鑰（大小寫要完全一樣）")
    print("   2. 該密鑰的「筆記本存取權」開關是否已打開")
    raise

# 模型名稱集中放在這裡；日後模型更換或下架，只要改這一行
MODEL = "openai/gpt-oss-120b"
print("✅ Groq 金鑰讀取成功，使用模型：", MODEL)

✅ Groq 金鑰讀取成功，使用模型： openai/gpt-oss-120b


In [9]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================
import sqlite3
import pandas as pd
from IPython.display import display

DB = "clinic.db"
DEPARTMENTS = ["內科", "外科", "牙科"]
TOTAL_QUOTA = 30  # 每科每日固定名額

conn = sqlite3.connect(DB)
cursor = conn.cursor()

# 每次執行先刪除舊表，確保從頭跑結果一致
cursor.execute("DROP TABLE IF EXISTS clinic_status")
cursor.execute("DROP TABLE IF EXISTS appointments")

# 1. 建立科別狀態表
cursor.execute("""
    CREATE TABLE clinic_status (
        department   TEXT PRIMARY KEY,
        total_quota  INTEGER NOT NULL,
        booked_count INTEGER NOT NULL
    )
""")

# 2. 建立掛號紀錄表
cursor.execute("""
    CREATE TABLE appointments (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        department  TEXT NOT NULL,
        status      TEXT NOT NULL,
        created_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

# 放入初始資料（內、外、牙科各 30 名額，已掛號 0）
initial_data = [(dept, TOTAL_QUOTA, 0) for dept in DEPARTMENTS]
cursor.executemany(
    "INSERT INTO clinic_status (department, total_quota, booked_count) VALUES (?, ?, ?)",
    initial_data
)

conn.commit()
conn.close()
print("✅ 診所資料庫與初始資料建立完成")

✅ 診所資料庫與初始資料建立完成


In [10]:
# ============================================================
# 第 3 格：顯示資料表
# ============================================================
def show_tables():
    conn = sqlite3.connect(DB)
    print("📦 clinic_status（科別狀態表）")
    display(pd.read_sql_query("SELECT * FROM clinic_status", conn))
    print("📝 appointments（掛號記錄表）")
    display(pd.read_sql_query("SELECT * FROM appointments", conn))
    conn.close()

show_tables()

📦 clinic_status（科別狀態表）


,department,total_quota,booked_count
0,內科,30,0
1,外科,30,0
2,牙科,30,0


📝 appointments（掛號記錄表）


,id,department,status,created_time


In [11]:
# ============================================================
# 第 4 格：AI 解讀函式
# ============================================================
ACTIONS = ["掛號", "取消", "查詢"]
dept_list_str = "、".join(DEPARTMENTS)

def ai_parse(user_say):
    prompt = f"""你是診所掛號助理系統。
本院科別僅限：{dept_list_str}。
功能僅限：掛號、取消、查詢剩餘名額。

🔴 安全規定：系統絕對不可以提供任何醫療建議、診斷或根據症狀推薦科別。
如果使用者只描述身體症狀而沒有明確說出科別（例如「肚子痛」、「頭暈想看醫生」），動作與對象一律回傳「未知,未知」。

請判斷使用者的「動作」與「對象（科別）」，只能回覆一行，格式為：動作,對象
- 動作只能是：掛號、取消、查詢、未知
- 對象只能是：{dept_list_str}、未知
- 不要回覆任何其他文字

範例：
使用者：我想掛內科 → 掛號,內科
使用者：幫我取消外科 → 取消,外科
使用者：牙科還有名額嗎 → 查詢,牙科
使用者：胸悶想看醫生 → 未知,未知

使用者：{user_say}"""

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        raw = response.choices[0].message.content or ""
    except Exception as e:
        print("⚠️ 呼叫 AI 失敗：", e)
        return "未知", "未知", ""

    # 防呆檢查：從 AI 回覆中比對合法的動作與科別
    action = "未知"
    for a in ACTIONS:
        if a in raw:
            action = a
            break

    department = "未知"
    for d in DEPARTMENTS:
        if d in raw:
            department = d
            break

    return action, department, raw

In [12]:
# ============================================================
# 第 5 格：測試清單（至少 8 句）
# ============================================================
test_list = [
    ("我要掛內科",           "掛號", "內科"),     # 正常
    ("幫我預約牙科",         "掛號", "牙科"),     # 正常
    ("查詢外科剩餘名額",     "查詢", "外科"),     # 正常
    ("取消外科掛號",         "取消", "外科"),     # 正常
    ("我想看醫生",           "未知", "未知"),     # 錯誤／邊界：未指定科別
    ("我頭暈想吐該看哪一科？", "未知", "未知"),     # 錯誤／邊界：醫療建議詢問
    ("我要掛眼科",           "未知", "未知"),     # 錯誤／邊界：不支援的科別
    ("我想掛內科",           "掛號", "內科"),     # 邊界測試
]

passed = 0
for say, exp_action, exp_item in test_list:
    action, item, raw = ai_parse(say)
    ok = (exp_action == action) and (exp_item == item)
    passed += ok
    print(f"{'✅' if ok else '❌'} {say}")
    print(f"     AI 原始回覆：{raw!r}")
    print(f"     系統解讀：{action} / {item}（預期：{exp_action} / {exp_item}）\n")

print(f"通過 {passed} / {len(test_list)} 句")

✅ 我要掛內科
     AI 原始回覆：'掛號,內科'
     系統解讀：掛號 / 內科（預期：掛號 / 內科）

✅ 幫我預約牙科
     AI 原始回覆：'掛號,牙科'
     系統解讀：掛號 / 牙科（預期：掛號 / 牙科）

✅ 查詢外科剩餘名額
     AI 原始回覆：'查詢,外科'
     系統解讀：查詢 / 外科（預期：查詢 / 外科）

✅ 取消外科掛號
     AI 原始回覆：'取消,外科'
     系統解讀：取消 / 外科（預期：取消 / 外科）

✅ 我想看醫生
     AI 原始回覆：'未知,未知'
     系統解讀：未知 / 未知（預期：未知 / 未知）

✅ 我頭暈想吐該看哪一科？
     AI 原始回覆：'未知,未知'
     系統解讀：未知 / 未知（預期：未知 / 未知）

❌ 我要掛眼科
     AI 原始回覆：'掛號,未知'
     系統解讀：掛號 / 未知（預期：未知 / 未知）

✅ 我想掛內科
     AI 原始回覆：'掛號,內科'
     系統解讀：掛號 / 內科（預期：掛號 / 內科）

通過 7 / 8 句


In [13]:
# ============================================================
# 第 5 格：測試清單（至少 8 句）
# ============================================================
test_list = [
    ("我要掛內科",           "掛號", "內科"),     # 正常
    ("幫我預約牙科",         "掛號", "牙科"),     # 正常
    ("查詢外科剩餘名額",     "查詢", "外科"),     # 正常
    ("取消外科掛號",         "取消", "外科"),     # 正常
    ("我想看醫生",           "未知", "未知"),     # 錯誤／邊界：未指定科別
    ("我頭暈想吐該看哪一科？", "未知", "未知"),     # 錯誤／邊界：醫療建議詢問
    ("我要掛眼科",           "未知", "未知"),     # 錯誤／邊界：不支援的科別
    ("我想掛內科",           "掛號", "內科"),     # 邊界測試
]

passed = 0
for say, exp_action, exp_item in test_list:
    action, item, raw = ai_parse(say)
    ok = (exp_action == action) and (exp_item == item)
    passed += ok
    print(f"{'✅' if ok else '❌'} {say}")
    print(f"     AI 原始回覆：{raw!r}")
    print(f"     系統解讀：{action} / {item}（預期：{exp_action} / {exp_item}）\n")

print(f"通過 {passed} / {len(test_list)} 句")

✅ 我要掛內科
     AI 原始回覆：'掛號,內科'
     系統解讀：掛號 / 內科（預期：掛號 / 內科）

✅ 幫我預約牙科
     AI 原始回覆：'掛號,牙科'
     系統解讀：掛號 / 牙科（預期：掛號 / 牙科）

✅ 查詢外科剩餘名額
     AI 原始回覆：'查詢,外科'
     系統解讀：查詢 / 外科（預期：查詢 / 外科）

✅ 取消外科掛號
     AI 原始回覆：'取消,外科'
     系統解讀：取消 / 外科（預期：取消 / 外科）

✅ 我想看醫生
     AI 原始回覆：'未知,未知'
     系統解讀：未知 / 未知（預期：未知 / 未知）

✅ 我頭暈想吐該看哪一科？
     AI 原始回覆：'未知,未知'
     系統解讀：未知 / 未知（預期：未知 / 未知）

❌ 我要掛眼科
     AI 原始回覆：'掛號,未知'
     系統解讀：掛號 / 未知（預期：未知 / 未知）

✅ 我想掛內科
     AI 原始回覆：'掛號,內科'
     系統解讀：掛號 / 內科（預期：掛號 / 內科）

通過 7 / 8 句


In [14]:
# ============================================================
# 第 6 格：功能函式（掛號、取消、查詢）
# ============================================================
def handle_clinic(action, department):
    conn = sqlite3.connect(DB)
    try:
        cursor = conn.cursor()
        cursor.execute("SELECT total_quota, booked_count FROM clinic_status WHERE department = ?", (department,))
        row = cursor.fetchone()

        if row is None:
            return f"系統查無「{department}」此科別。"

        total_quota, booked_count = row

        if action == "查詢":
            left = total_quota - booked_count
            return f"【{department}】今日總名額 {total_quota} 位，目前已掛 {booked_count} 位，剩餘名額還有 {left} 位。"

        elif action == "掛號":
            if booked_count >= total_quota:
                return f"非常抱歉，【{department}】今日名額已滿。"

            # 更新已掛號人數並新增預約紀錄
            cursor.execute("UPDATE clinic_status SET booked_count = booked_count + 1 WHERE department = ?", (department,))
            cursor.execute("INSERT INTO appointments (department, status) VALUES (?, ?)", (department, "已掛號"))
            conn.commit()
            return f"🎉 成功為您完成【{department}】掛號！"

        elif action == "取消":
            # 檢查是否有該科別的有效掛號
            cursor.execute("SELECT id FROM appointments WHERE department = ? AND status = '已掛號' ORDER BY id DESC LIMIT 1", (department,))
            app_row = cursor.fetchone()
            if app_row is None:
                return f"查無【{department}】的有效掛號記錄，無法取消。"

            cursor.execute("UPDATE clinic_status SET booked_count = MAX(0, booked_count - 1) WHERE department = ?", (department,))
            cursor.execute("UPDATE appointments SET status = '已取消' WHERE id = ?", (app_row[0],))
            conn.commit()
            return f"✅ 已為您取消【{department}】的掛號。"

        return "無法識別的操作指令。"
    finally:
        conn.close()


def clinic_system(user_say):
    if not user_say or not user_say.strip():
        return "請輸入您的掛號或查詢需求。"

    action, department, raw = ai_parse(user_say)
    print(f"📌 [AI 解讀] 動作={action}，對象={department}（原始回覆：{raw!r}）")

    if department == "未知" or action == "未知":
        return "本院僅提供內科、外科、牙科服務，且無法根據症狀推薦科別。請明確告訴我要掛哪一科，或請洽詢櫃台。"

    return handle_clinic(action, department)

In [15]:
# ============================================================
# 第 7 格：主流程與完整流程測試
# ============================================================
test_cases = [
    "我要掛內科",
    "牙科還有名額嗎",
    "幫我掛牙科",
    "取消牙科掛號",
    "我想看醫生",
    "肚子痛該看哪一科",
]

for say in test_cases:
    print("-" * 50)
    print(f"🙋 使用者：{say}")
    print(f"🤖 系統回應：{clinic_system(say)}")

print("-" * 50)
show_tables()

--------------------------------------------------
🙋 使用者：我要掛內科
📌 [AI 解讀] 動作=掛號，對象=內科（原始回覆：'掛號,內科'）
🤖 系統回應：🎉 成功為您完成【內科】掛號！
--------------------------------------------------
🙋 使用者：牙科還有名額嗎
📌 [AI 解讀] 動作=查詢，對象=牙科（原始回覆：'查詢,牙科'）
🤖 系統回應：【牙科】今日總名額 30 位，目前已掛 0 位，剩餘名額還有 30 位。
--------------------------------------------------
🙋 使用者：幫我掛牙科
📌 [AI 解讀] 動作=掛號，對象=牙科（原始回覆：'掛號,牙科'）
🤖 系統回應：🎉 成功為您完成【牙科】掛號！
--------------------------------------------------
🙋 使用者：取消牙科掛號
📌 [AI 解讀] 動作=取消，對象=牙科（原始回覆：'取消,牙科'）
🤖 系統回應：✅ 已為您取消【牙科】的掛號。
--------------------------------------------------
🙋 使用者：我想看醫生
📌 [AI 解讀] 動作=未知，對象=未知（原始回覆：'未知,未知'）
🤖 系統回應：本院僅提供內科、外科、牙科服務，且無法根據症狀推薦科別。請明確告訴我要掛哪一科，或請洽詢櫃台。
--------------------------------------------------
🙋 使用者：肚子痛該看哪一科
📌 [AI 解讀] 動作=未知，對象=未知（原始回覆：'未知,未知'）
🤖 系統回應：本院僅提供內科、外科、牙科服務，且無法根據症狀推薦科別。請明確告訴我要掛哪一科，或請洽詢櫃台。
--------------------------------------------------
📦 clinic_status（科別狀態表）


,department,total_quota,booked_count
0,內科,30,1
1,外科,30,0
2,牙科,30,0


📝 appointments（掛號記錄表）


,id,department,status,created_time
0,1,內科,已掛號,2026-10-29 04:00:29
1,2,牙科,已取消,2026-10-29 04:00:30


In [16]:
# ============================================================
# 第 8 格：Gradio 網頁
# ============================================================
demo = gr.Interface(
    fn=clinic_system,
    inputs=gr.Textbox(label="💬 請輸入您的需求（例如：我要掛內科、查詢牙科名額）"),
    outputs=gr.Textbox(label="🤖 診所系統回應"),
    title="🏥 智能診所掛號系統",
    description="支援內科、外科、牙科之掛號、取消與剩餘名額查詢（嚴格遵守不提供醫療建議規範）",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://36bea3e5aaf41fbaa1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
